# Improved Low-level Linear Model in Pytorch

This version has the following changes/improvements over the `basic` version:

1. Allow for GPU support by checking for availability and moving tensors to the respective device.
2. Stabilize training numerically by using `log_softmax` directly.
3. Refactor model parameters into a list to make training more easily extendible to more layers (since we can loop over the list for gradient descent).

To avoid copy & paste errors, this notebook misses most explanations for the individual steps compared to the basic one. So check that one for more details.

In [ ]:
# shows GPU usage (mostly useful in mutli-GPU setups)
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from time import perf_counter

import numpy as np
import torch
from matplotlib import pyplot as plt

from idl.common import accuracy
from idl.week1.data import get_mnist, load_mnist, mnist_overview
from idl.week1.analysis import confusion_matrix, plot_learning_curves, precision_recall

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device is", DEVICE)

## Getting Data

In [ ]:
get_mnist()
(x_train, y_train), (x_valid, y_valid), (x_test, y_test) = load_mnist()

## Building a Model

This block contains the main differences to the basic notebook.

In [ ]:
class Model:
    def __init__(self):
        # DIFFERENCE parameters as list
        self.parameters = [torch.zeros(10, 784, requires_grad=True),
                           torch.zeros(10, requires_grad=True)]
        # example to create random weights instead.
        # you NEED this for MLPs! for linear models, 0s are fine.
        # biases are usually fine to initialize with 0s even for MLPs.
        #weights = torch.distributions.Uniform(-0.01, 0.01).sample((10, 784))

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        logits = inputs @ self.parameters[0].transpose(0, 1) + self.parameters[1]
        # DIFFERENCE using log softmax
        return log_softmax(logits)

    def __call__(self,
                 inputs: torch.Tensor) -> torch.Tensor:
        return self.forward(inputs)

    # DIFFERENCE new function to move parameters to device
    def to(self, device: str):
        with torch.no_grad():
            for ind in range(len(self.parameters)):
                self.parameters[ind] = self.parameters[ind].to(device)
                self.parameters[ind].requires_grad = True

    @property
    def device(self):
        return self.parameters[0].device


# DIFFERENCE
def log_softmax(inputs: torch.Tensor) -> torch.Tensor:
    return inputs - torch.logsumexp(inputs, dim=1, keepdims=True)


# DIFFERENCE using log softmax outputs directly
def cross_entropy(log_outputs: torch.Tensor,
                  labels: torch.Tensor) -> torch.Tensor:
    labels_one_hot = one_hot(labels, num_classes=10)
    return -(labels_one_hot * log_outputs).sum(axis=-1).mean()


def one_hot(inputs: torch.Tensor,
            num_classes: int | None = None) -> torch.Tensor:
    if num_classes is None:
        num_classes = inputs.max() + 1
    output = torch.zeros(inputs.shape[0], num_classes, device=inputs.device)
    output.scatter_(1, inputs.view(-1, 1), torch.ones(inputs.shape[0], 1, device=inputs.device))

    return output

In [ ]:
linear_model = Model()
with torch.no_grad():
    print(linear_model(torch.tensor(x_train[:4])))

In [ ]:
with torch.no_grad():
    print(cross_entropy(linear_model(torch.tensor(x_train[:4])), torch.tensor(y_train[:4])))
np.log(10)

In [ ]:
with torch.no_grad():
    print(accuracy(linear_model(torch.tensor(x_train[:4])), torch.tensor(y_train[:4])))

In [ ]:
# NOTE! if running on GPU, after running this cell, 
# the above tests will no longer work because the weights are on GPU,
# but the input tensors on CPU.
# you would have to move the input tensors to GPU a well!
linear_model.to(DEVICE)
print(linear_model.parameters)

## Training

Here we have some more differences to the basic notebook; mainly moving the dataset to the correct device.
Aside from that, this is copy & paste. This could obviously be improved by moving the training loop to separate Python file and importing.

In [ ]:
def train_linear_model(model: Model, 
                       training_data: tuple[np.ndarray, np.ndarray], 
                       validation_data: tuple[np.ndarray, np.ndarray],
                       learning_rate: float,
                       batch_size: int,
                       n_epochs: int,
                       verbose: bool = True) -> dict[str, np.ndarray]:
    n_training_examples = len(training_data[0])
    batches_per_epoch = n_training_examples // batch_size
    print("Running {} epochs at {} steps per epoch.".format(n_epochs, batches_per_epoch))
    
    random_shuffling = np.arange(n_training_examples)

    # this moves the WHOLE dataset to the GPU!
    # this is somewhat wasteful in terms of memory and not recommended for larger datasets.
    # but MNIST is pretty small, so it's ok.
    # normally you should only move the batches to the GPU as you need them.
    # but that would be slightly slower, as copying the data takes time during the training loop.
    train_input_tensor = torch.tensor(training_data[0]).to(model.device)
    train_label_tensor = torch.tensor(training_data[1]).to(model.device)
    valid_input_tensor = torch.tensor(validation_data[0]).to(model.device)
    valid_label_tensor = torch.tensor(validation_data[1]).to(model.device)

    # note, for training we only track the average over the epoch.
    # this is somewhat imprecise, as the model changes over the epoch.
    # so the metrics at the end of the epoch will usually be better than at the start,
    # but we average over everything.
    # we could record train metrics more often to get a better picture of training progress.
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(n_epochs):
        if verbose:
            print("Starting epoch {}...".format(epoch + 1), end=" ")
        start_time = perf_counter()
        # every epoch we shuffle the data!
        np.random.shuffle(random_shuffling)  # this shuffles in place
        slice_index = 0
        shuffled_inputs = train_input_tensor[random_shuffling]
        shuffled_labels = train_label_tensor[random_shuffling]

        epoch_train_losses = []
        epoch_train_accuracies = []
        
        for batch_ind in range(batches_per_epoch):
            input_batch = shuffled_inputs[slice_index:slice_index+batch_size]
            label_batch = shuffled_labels[slice_index:slice_index+batch_size]
            # this is the core of model training!
            # all the other stuff is pretty much just providing data and recording metrics...
            # two lines "forward" to compute outputs and the loss
            output_batch = model(input_batch)
            batch_loss = cross_entropy(output_batch, label_batch)
            # and then backpropagate
            batch_loss.backward()
            with torch.no_grad():
                # DIFFERENCE loop over parameters
                for parameter in model.parameters:
                    parameter.sub_(parameter.grad * learning_rate)
                    parameter.grad.zero_()

            batch_accuracy = accuracy(output_batch, label_batch)
            epoch_train_losses.append(batch_loss.item())
            epoch_train_accuracies.append(batch_accuracy.item())
            
            slice_index += batch_size
        end_time = perf_counter()
        time_taken = end_time - start_time
            
        # evaluate after each epoch
        with torch.no_grad():
            val_output = model(valid_input_tensor)
            val_loss = cross_entropy(val_output, valid_label_tensor)
            val_accuracy = accuracy(val_output, valid_label_tensor)
        val_losses.append(val_loss.item())
        val_accuracies.append(val_accuracy.item())

        train_losses.append(np.mean(epoch_train_losses))
        train_accuracies.append(np.mean(epoch_train_accuracies))

        if verbose:
            print(f"Time taken: {time_taken:.3f} seconds")
            print(f"\tTrain/val loss: {train_losses[-1]:.5g} / {val_losses[-1]:.5g}")
            print(f"\tTrain/val accuracy: {train_accuracies[-1]:.5g} / {val_accuracies[-1]:.5g}")
            print()

    return {"train_loss": np.array(train_losses), "train_accuracy": np.array(train_accuracies),
            "val_loss": np.array(val_losses), "val_accuracy": np.array(val_accuracies)}

In [ ]:
metrics = train_linear_model(linear_model, (x_train, y_train), (x_valid, y_valid),
                             learning_rate=0.1, batch_size=128, n_epochs=15)

## Inspecting Model Performance

In [ ]:
plot_learning_curves(metrics, keys=["loss", "accuracy"])

In [ ]:
training_matrix = confusion_matrix(linear_model, x_train, y_train, device=DEVICE)
np.set_printoptions(suppress=True)
print("Training confusion matrix")
print(training_matrix)

validation_matrix = confusion_matrix(linear_model, x_valid, y_valid, device=DEVICE)
print()
print("Validation confusion matrix")
print(validation_matrix)

In [ ]:
print("TRAINING")
precision_recall(training_matrix)

print("\n\nVALIDATION")
precision_recall(validation_matrix)

In [ ]:
plt.figure(figsize=(12, 4))
for ind, pattern in enumerate(linear_model.parameters[0].detach().cpu().numpy()):
    plt.subplot(2, 5, ind+1)
    absmax = abs(pattern).max()
    plt.imshow(pattern.reshape(28, 28), vmin=-absmax, vmax=absmax, cmap="RdBu_r")
    plt.axis("off")
    plt.colorbar()
    plt.title(f"Pattern class {ind}")
plt.show()

In [ ]:
plt.bar(np.arange(10), linear_model.parameters[1].detach().cpu().numpy())
plt.xticks(np.arange(10))
plt.title("Bias for each label")
plt.show()

## Basic Hyperparameter Study: Learning Rate

In [ ]:
# to cover a large space, we commonly use so-called geometric progressions.
# this means there is a common *multipilicative* factor between successive values.
# here, each learning rate is roughly 3x the one before.
learning_rates = np.geomspace(0.0000001, 10., 17)
learning_rates

In [ ]:
all_metrics = {}
for lr_ind, learning_rate in enumerate(learning_rates):
    linear_model = Model()
    linear_model.to(DEVICE)
    print(f"Training learning rate {learning_rate}...")
    metrics = train_linear_model(linear_model, (x_train, y_train), (x_valid, y_valid),
                                 learning_rate=learning_rate, batch_size=128, n_epochs=10,
                                 verbose=False)
    all_metrics[lr_ind] = metrics

In [ ]:
# collect the final results only
final_metrics = {"val_loss": [], "train_loss": [], "val_accuracy": [], "train_accuracy": []}
for ind in range(len(learning_rates)):
    lr_results = all_metrics[ind]
    for key in lr_results:
        final_metrics[key].append(lr_results[key][-1])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(final_metrics["train_loss"], label="train")
plt.plot(final_metrics["val_loss"], label="validation")
plt.legend()
plt.title("Loss")
plt.xticks(np.arange(len(learning_rates)), np.round(learning_rates, 7), rotation="vertical")
plt.xlabel("Learning rate")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(final_metrics["train_accuracy"], label="train")
plt.plot(final_metrics["val_accuracy"], label="validation")
plt.legend()
plt.title("Accuracy")
plt.xticks(np.arange(len(learning_rates)), np.round(learning_rates, 7), rotation="vertical")
plt.xlabel("Learning rate")
plt.show()

In [ ]:
for key in lr_results:
    plt.figure(figsize=(10, 4))
    for ind, lr in enumerate(learning_rates):
        lr_results = all_metrics[ind]
        plt.plot(lr_results[key], label=str(np.round(lr, 7)))
    plt.title(key)
    plt.legend()
    plt.xlabel("Epoch")
    plt.show()